In [ ]:
import numpy as np
import pandas as pd

homes = pd.read_csv("../data/homes_clean.csv", parse_dates=["sold"])
homes["area"] = homes["area"].str.title()
homes["months"] = (homes["sold"].dt.year - 2021) * 12 + homes["sold"].dt.month - 1

In [ ]:
features = ["sqft", "beds", "built", "garage", "condition", "area", "months"]
X = homes[features]
y = homes["price"]
X.shape, y.shape

In [ ]:
X.head(3)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1460
)
len(X_train), len(X_test)

In [ ]:
y_train.median(), y_test.median()

In [ ]:
from sklearn.linear_model import LinearRegression

def log_features(X, y):
    return pd.DataFrame({
        "log_sqft": np.log(X["sqft"]),
        "log_per_sqft": np.log(y / X["sqft"]),
    })

leaky = LinearRegression().fit(log_features(X_train, y_train), np.log(y_train))
round(leaky.score(log_features(X_test, y_test), np.log(y_test)), 4)

In [ ]:
honest = LinearRegression().fit(np.log(X_train[["sqft"]]), np.log(y_train))
round(honest.score(np.log(X_test[["sqft"]]), np.log(y_test)), 4)

In [ ]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False).fit(X_train[["area"]])
encoder.get_feature_names_out()

In [ ]:
pd.DataFrame(
    encoder.transform(X_train[["area"]].head(3)),
    columns=encoder.get_feature_names_out(),
    index=X_train["area"].head(3),
)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(X_train[["sqft", "built"]])
scaler.mean_.round(1), scaler.scale_.round(1)

In [ ]:
scaled = scaler.transform(X_test[["sqft", "built"]])
pd.DataFrame(scaled, columns=["sqft", "built"]).agg(["mean", "std"]).round(3)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

numeric = ["sqft", "beds", "built", "garage", "condition", "months"]
numbers = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
prepare = ColumnTransformer([
    ("numbers", numbers, numeric),
    ("area", OneHotEncoder(), ["area"]),
])
prepare

In [ ]:
ready = prepare.fit_transform(X_train)
ready.shape, prepare.get_feature_names_out().tolist()

In [ ]:
from sklearn.metrics import mean_absolute_error

guess = np.full(len(y_test), y_train.median())
round(mean_absolute_error(y_test, guess))

In [ ]:
line = LinearRegression().fit(X_train[["sqft"]], y_train)
predicted = line.predict(X_test[["sqft"]])
error = mean_absolute_error(y_test, predicted)
round(line.intercept_), round(line.coef_[0], 1), round(error)

In [ ]:
ax = X_test.assign(price=y_test).plot.scatter(x="sqft", y="price", s=4, alpha=0.4)
ends = pd.DataFrame({"sqft": [500, 4000]})
ax.plot(ends["sqft"], line.predict(ends), color="red");

In [ ]:
X_test[["sqft"]].assign(
    actual=y_test,
    predicted=predicted.round(-2),
    error=(y_test - predicted).round(-2),
).head(4)

In [ ]:
def rmse_for(slope):
    intercept = y_train.mean() - slope * X_train["sqft"].mean()
    errors = y_train - (intercept + slope * X_train["sqft"])
    return np.sqrt((errors ** 2).mean())

slopes = np.arange(100, 181, 2)
errors = pd.Series([rmse_for(s) for s in slopes], index=slopes)
errors.plot(title="error for each slope");

In [ ]:
plain = ColumnTransformer([
    ("numbers", SimpleImputer(strategy="median"), numeric),
    ("area", OneHotEncoder(drop=["Eastfield"]), ["area"]),
])
linear = make_pipeline(plain, LinearRegression()).fit(X_train, y_train)
round(mean_absolute_error(y_test, linear.predict(X_test)))

In [ ]:
names = plain.get_feature_names_out()
pd.Series(linear[-1].coef_, index=names).round()

In [ ]:
from sklearn.compose import TransformedTargetRegressor

log_linear = TransformedTargetRegressor(
    make_pipeline(plain, LinearRegression()), func=np.log, inverse_func=np.exp
).fit(X_train, y_train)

def typical_miss(model):
    """The median error as a share of the price."""
    return round((abs(model.predict(X_test) - y_test) / y_test).median(), 3)

typical_miss(linear), typical_miss(log_linear)

In [ ]:
by_price = pd.qcut(y_test, 3, labels=["cheapest third", "middle", "dearest third"])
pd.DataFrame({
    "linear": (linear.predict(X_test) - y_test).abs().groupby(by_price).median(),
    "log": (log_linear.predict(X_test) - y_test).abs().groupby(by_price).median(),
}).round(-2)

In [ ]:
from sklearn.tree import DecisionTreeRegressor, export_text

tree = make_pipeline(plain, DecisionTreeRegressor(max_depth=3, random_state=1460))
tree.fit(X_train, y_train)
round(mean_absolute_error(y_test, tree.predict(X_test)))

In [ ]:
print(export_text(tree[-1], feature_names=list(names), decimals=0))

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

boosted = make_pipeline(plain, HistGradientBoostingRegressor(random_state=1460))
boosted.fit(X_train, y_train)

def mae(model, X=X_test, y=y_test):
    return round(mean_absolute_error(y, model.predict(X)))

pd.Series({
    "median guess": round(mean_absolute_error(y_test, guess)),
    "line on sqft": round(mean_absolute_error(y_test, predicted)),
    "linear": mae(linear),
    "log linear": mae(log_linear),
    "tree, depth 3": mae(tree),
    "boosted trees": mae(boosted),
})

In [ ]:
rows = []
for depth in range(1, 21):
    model = DecisionTreeRegressor(max_depth=depth, random_state=1460)
    deep = make_pipeline(plain, model).fit(X_train, y_train)
    train_error = mae(deep, X_train, y_train)
    rows.append({"depth": depth, "train": train_error, "test": mae(deep)})
depths = pd.DataFrame(rows).set_index("depth")
depths.loc[[1, 3, 6, 10, 20]]

In [ ]:
depths.plot(title="mean absolute error by tree depth");

In [ ]:
from sklearn.model_selection import KFold, cross_val_score

folds = KFold(5, shuffle=True, random_state=1460)
scores = -cross_val_score(
    linear, X_train, y_train, cv=folds, scoring="neg_mean_absolute_error"
)
scores.round(), round(scores.mean()), round(scores.std())

In [ ]:
from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(
    boosted,
    {
        "histgradientboostingregressor__learning_rate": [0.03, 0.1, 0.3],
        "histgradientboostingregressor__max_leaf_nodes": [7, 15, 31],
    },
    cv=folds,
    scoring="neg_mean_absolute_error",
).fit(X_train, y_train)

In [ ]:
results = pd.DataFrame(grid.cv_results_)
param = "param_histgradientboostingregressor__"
pd.DataFrame({
    "learning_rate": results[param + "learning_rate"],
    "max_leaf_nodes": results[param + "max_leaf_nodes"],
    "cv_mae": -results["mean_test_score"].round(),
}).sort_values("cv_mae").head(4)

In [ ]:
round(-grid.best_score_), round(scores.mean()), mae(grid.best_estimator_), mae(linear)

In [ ]:
residual = y_test - linear.predict(X_test)
pd.DataFrame({
    "median error": residual.groupby(X_test["area"]).median(),
    "median miss": residual.abs().groupby(X_test["area"]).median(),
}).round(-2)

In [ ]:
residual.plot.hist(bins=30, title="test errors: actual − predicted");

In [ ]:
from sklearn.inspection import permutation_importance

importance = permutation_importance(
    linear, X_test, y_test, n_repeats=20, random_state=1460,
    scoring="neg_mean_absolute_error",
)
pd.Series(importance.importances_mean, index=X_test.columns).sort_values().round()

In [ ]:
def quantile_model(q):
    booster = HistGradientBoostingRegressor(
        loss="quantile", quantile=q, random_state=1460
    )
    return make_pipeline(plain, booster).fit(X_train, y_train)

bands = {q: quantile_model(q) for q in (0.1, 0.5, 0.9)}
house = X_test.head(1)
house

In [ ]:
{q: round(m.predict(house)[0], -3) for q, m in bands.items()}, y_test.iloc[0]

In [ ]:
low, high = bands[0.1].predict(X_test), bands[0.9].predict(X_test)
round(((y_test >= low) & (y_test <= high)).mean(), 3)

In [ ]:
from sklearn.model_selection import cross_val_predict

unseen = y_train - cross_val_predict(linear, X_train, y_train, cv=folds)
below, above = np.quantile(unseen, [0.1, 0.9])
point = linear.predict(X_test)
inside = (y_test >= point + below) & (y_test <= point + above)
round(below, -2), round(above, -2), round(inside.mean(), 3)